# 11. 역전파 직접 구현 ★

> **제11장** · **이론편 대응: 5.3절(연쇄법칙), 10.4절(역전파)**
> **예상 소요**: 70분
> **필요 사양**: CPU만으로 충분

---

## 이 장이 특별한 이유

이론편 10.4절에서 **2-2-1 신경망의 역전파를 손으로 끝까지 계산**했다.
이 장에서 그 계산을 코드로 다시 하고, **소수점까지 같은지 확인한다.**

이것이 이론편과 2권을 잇는 가장 중요한 지점이다. 이론과 구현이 같은 것을 말하고 있다는 사실을
직접 눈으로 확인하는 경험은, 두 권을 따로 읽어서는 얻을 수 없다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 순전파 — 이론편 값 검증 | 10.4절 |
| 2 | 역전파를 한 항씩 유도 | 5.3절 |
| 3 | **손계산 값 4개 대조** ★ | 10.4절 |
| 4 | 수치 미분으로 3중 검증 | 05장 |
| 5 | 그래디언트 소실 확인 | 11.2절 |
| 6 | 일반화 — 층 개수에 무관한 구현 | — |

### 이론편 10.4절에서 손으로 구한 값

| 항목 | 값 |
|---|---|
| $z^{(1)}$ | (0.40, 0.25) |
| $a^{(1)}$ | (0.5987, 0.5622) |
| $z^{(2)}$ | 0.8652 |
| 손실 $L$ | 0.0182 |
| $\delta^{(2)}$ | −0.2697 |
| $\partial L/\partial W^{(2)}$ | (−0.1614, −0.1516) |
| $\partial L/\partial W^{(1)}$ | (−0.0389, −0.0194 / −0.0597, −0.0299) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_derivative(a):
    """a는 sigmoid의 출력값 (이론편 10.3절에서 유도)"""
    return a * (1.0 - a)

print("준비 완료")

---

## 1. 순전파 — 이론편 10.4절 값 검증

이론편 10.4절의 설정을 그대로 옮긴다.

$$\mathbf{x} = \begin{pmatrix} 1.0 \\ 0.5 \end{pmatrix}, \quad
W^{(1)} = \begin{pmatrix} 0.2 & 0.4 \\ 0.1 & 0.3 \end{pmatrix}, \quad
W^{(2)} = \begin{pmatrix} 0.6 & 0.9 \end{pmatrix}, \quad y = 1.0$$

편향은 생략했다(이론편과 동일). 순전파는 다음 순서다.

$$\mathbf{z}^{(1)} = W^{(1)}\mathbf{x} \;\to\; \mathbf{a}^{(1)} = \sigma(\mathbf{z}^{(1)}) \;\to\; z^{(2)} = W^{(2)}\mathbf{a}^{(1)} \;\to\; L = (z^{(2)}-y)^2$$

In [ ]:
import numpy as np

# 이론편 10.4절과 완전히 같은 값
x  = np.array([1.0, 0.5])
W1 = np.array([[0.2, 0.4],
               [0.1, 0.3]])
W2 = np.array([0.6, 0.9])
y  = 1.0

print("=" * 60)
print("순전파 — 이론편 10.4절 값 검증")
print("=" * 60)

# 1단계: 은닉층 입력
z1 = W1 @ x
print(f"z1 = W1 @ x")
print(f"   = [{W1[0,0]}*{x[0]} + {W1[0,1]}*{x[1]},  {W1[1,0]}*{x[0]} + {W1[1,1]}*{x[1]}]")
print(f"   = {z1.round(4)}          이론편: [0.40, 0.25]")

# 2단계: 활성화
a1 = sigmoid(z1)
print(f"\na1 = sigmoid(z1)")
print(f"   = {a1.round(4)}      이론편: [0.5987, 0.5622]")

# 3단계: 출력층
z2 = W2 @ a1
print(f"\nz2 = W2 @ a1")
print(f"   = {W2[0]}*{a1[0]:.4f} + {W2[1]}*{a1[1]:.4f}")
print(f"   = {z2:.4f}                    이론편: 0.8652")

# 4단계: 손실
L = (z2 - y) ** 2
print(f"\nL  = (z2 - y)^2 = ({z2:.4f} - {y})^2")
print(f"   = {L:.4f}                    이론편: 0.0182")

print("-" * 60)
assert np.allclose(z1, [0.40, 0.25], atol=1e-4)
assert np.allclose(a1, [0.5987, 0.5622], atol=1e-4)
assert abs(z2 - 0.8652) < 1e-4
assert abs(L - 0.0182) < 1e-4
print("[OK] 순전파 4개 값 모두 이론편 10.4절과 일치")

---

## 2. 역전파를 한 항씩 유도

이론편 5.3절의 연쇄법칙을 적용한다. **출력 쪽에서 시작해 거꾸로** 간다.

### 2-1. 출력에서 시작

손실 $L = (z^{(2)} - y)^2$을 $z^{(2)}$로 미분한다.

$$\frac{\partial L}{\partial z^{(2)}} = 2(z^{(2)} - y)$$

이 값을 $\delta^{(2)}$라 부른다. **뒤에서 앞으로 전달할 신호**라는 뜻이다.

### 2-2. 출력층 가중치

$z^{(2)} = W^{(2)}\mathbf{a}^{(1)}$이므로, $W^{(2)}$의 각 성분으로 미분하면 대응하는 $a^{(1)}$만 남는다.

$$\frac{\partial L}{\partial W^{(2)}_j} = \delta^{(2)} \cdot a^{(1)}_j$$

### 2-3. 은닉층으로 되돌리기

각 은닉 뉴런이 출력에 기여한 만큼 신호를 나눠 받고, 시그모이드 도함수를 곱한다.

$$\delta^{(1)}_j = \delta^{(2)} \cdot W^{(2)}_j \cdot a^{(1)}_j(1-a^{(1)}_j)$$

### 2-4. 입력층 가중치

$$\frac{\partial L}{\partial W^{(1)}_{ji}} = \delta^{(1)}_j \cdot x_i$$

**패턴이 보인다.** 각 층의 가중치 그래디언트는 항상 **"그 층으로 온 신호 × 그 층의 입력"**이다.

In [ ]:
import numpy as np

print("=" * 60)
print("역전파 — 단계별 계산")
print("=" * 60)

# --- 2-1. 출력에서 시작 ---
delta2 = 2 * (z2 - y)
print(f"[1단계] delta2 = 2(z2 - y) = 2({z2:.4f} - {y})")
print(f"        = {delta2:.4f}                 이론편: -0.2697")

# --- 2-2. 출력층 가중치 ---
grad_W2 = delta2 * a1
print(f"\n[2단계] dL/dW2 = delta2 * a1")
print(f"        = {delta2:.4f} * {a1.round(4)}")
print(f"        = {grad_W2.round(4)}     이론편: [-0.1614, -0.1516]")

# --- 2-3. 은닉층으로 되돌리기 ---
sig_deriv = sigmoid_derivative(a1)
delta1 = delta2 * W2 * sig_deriv
print(f"\n[3단계] a1(1-a1) = {sig_deriv.round(4)}   이론편: [0.2403, 0.2461]")
print(f"        delta1 = delta2 * W2 * a1(1-a1)")
print(f"        = {delta2:.4f} * {W2} * {sig_deriv.round(4)}")
print(f"        = {delta1.round(4)}     이론편: [-0.0389, -0.0597]")

# --- 2-4. 입력층 가중치 ---
grad_W1 = np.outer(delta1, x)
print(f"\n[4단계] dL/dW1 = delta1 (외적) x")
print(f"        =\n{grad_W1.round(4)}")
print(f"        이론편:")
print(f"        [[-0.0389 -0.0194]")
print(f"         [-0.0597 -0.0299]]")

### `np.outer`를 쓰는 이유

$\partial L/\partial W^{(1)}_{ji} = \delta^{(1)}_j \cdot x_i$ 는 **모든 $(j, i)$ 조합**에 대한 계산이다.

- $\delta^{(1)}$: 길이 2 (은닉 뉴런 수)
- $\mathbf{x}$: 길이 2 (입력 개수)
- 결과: 2×2 행렬

`np.outer(a, b)`는 `a[i] * b[j]`를 모든 조합에 대해 계산해 행렬로 만든다.
반복문 두 개를 쓰는 것과 같지만 훨씬 빠르다.

---

## 3. 손계산 값 대조 ★

이제 이론편 10.4절의 표와 나란히 놓고 확인한다.

In [ ]:
import numpy as np

print("=" * 70)
print("이론편 10.4절 손계산 값과 대조")
print("=" * 70)

checks = [
    ("z1[0]",        z1[0],        0.40),
    ("z1[1]",        z1[1],        0.25),
    ("a1[0]",        a1[0],        0.5987),
    ("a1[1]",        a1[1],        0.5622),
    ("z2",           z2,           0.8652),
    ("L",            L,            0.0182),
    ("delta2",       delta2,      -0.2697),
    ("dL/dW2[0]",    grad_W2[0],  -0.1614),
    ("dL/dW2[1]",    grad_W2[1],  -0.1516),
    ("a1(1-a1)[0]",  sig_deriv[0], 0.2403),
    ("a1(1-a1)[1]",  sig_deriv[1], 0.2461),
    ("delta1[0]",    delta1[0],   -0.0389),
    ("delta1[1]",    delta1[1],   -0.0597),
    ("dL/dW1[0][0]", grad_W1[0,0], -0.0389),
    ("dL/dW1[0][1]", grad_W1[0,1], -0.0194),
    ("dL/dW1[1][0]", grad_W1[1,0], -0.0597),
    ("dL/dW1[1][1]", grad_W1[1,1], -0.0299),
]

print(f"{'항목':<16}{'코드 계산':<16}{'이론편 손계산':<16}{'차이':<14}{'판정'}")
print("-" * 70)
all_ok = True
for name, calc, book in checks:
    diff = abs(calc - book)
    ok = diff < 5e-4
    all_ok &= ok
    print(f"{name:<16}{calc:<16.6f}{book:<16.4f}{diff:<14.2e}{'O' if ok else 'X'}")
print("-" * 70)

assert all_ok, "이론편 값과 일치하지 않는 항목이 있습니다"
print(f"[OK] {len(checks)}개 항목 전부 일치")
print()
print("이론과 구현이 같은 것을 말하고 있다는 사실을 확인했다.")

---

## 4. 수치 미분으로 3중 검증

이론편 값과 맞았다는 것만으로는 부족할 수 있다. 이론편의 계산 자체가 틀렸을 가능성도 있으니까.

**독립적인 방법으로 한 번 더 확인한다.** 05장에서 만든 수치 미분 검증을 쓴다.

$$\frac{\partial L}{\partial w} \approx \frac{L(w+h) - L(w-h)}{2h}$$

이 방법은 미분 공식을 전혀 쓰지 않고 **손실 값만으로** 기울기를 구한다.
따라서 우리가 유도한 식이 맞는지 독립적으로 확인해 준다.

In [ ]:
import numpy as np

def forward_loss(W1_, W2_, x_, y_):
    """가중치를 받아 손실만 돌려주는 함수 (수치 미분용)"""
    a1_ = sigmoid(W1_ @ x_)
    z2_ = W2_ @ a1_
    return (z2_ - y_) ** 2


def numeric_grad(W1_, W2_, x_, y_, h=1e-7):
    """중앙 차분으로 모든 가중치의 그래디언트를 구한다"""
    gW1 = np.zeros_like(W1_)
    for i in range(W1_.shape[0]):
        for j in range(W1_.shape[1]):
            Wp = W1_.copy(); Wp[i, j] += h
            Wm = W1_.copy(); Wm[i, j] -= h
            gW1[i, j] = (forward_loss(Wp, W2_, x_, y_) -
                         forward_loss(Wm, W2_, x_, y_)) / (2 * h)

    gW2 = np.zeros_like(W2_)
    for j in range(len(W2_)):
        Wp = W2_.copy(); Wp[j] += h
        Wm = W2_.copy(); Wm[j] -= h
        gW2[j] = (forward_loss(W1_, Wp, x_, y_) -
                  forward_loss(W1_, Wm, x_, y_)) / (2 * h)

    return gW1, gW2


num_W1, num_W2 = numeric_grad(W1, W2, x, y)

print("=" * 70)
print("3중 검증 — 수식 유도 / 이론편 손계산 / 수치 미분")
print("=" * 70)
print(f"{'항목':<16}{'수식 유도':<16}{'이론편':<12}{'수치 미분':<16}{'최대 차이'}")
print("-" * 70)

book_W1 = np.array([[-0.0389, -0.0194], [-0.0597, -0.0299]])
book_W2 = np.array([-0.1614, -0.1516])

for i in range(2):
    for j in range(2):
        name = f"dL/dW1[{i}][{j}]"
        d = max(abs(grad_W1[i,j] - num_W1[i,j]), abs(grad_W1[i,j] - book_W1[i,j]))
        print(f"{name:<16}{grad_W1[i,j]:<16.6f}{book_W1[i,j]:<12.4f}"
              f"{num_W1[i,j]:<16.6f}{d:.2e}")

for j in range(2):
    name = f"dL/dW2[{j}]"
    d = max(abs(grad_W2[j] - num_W2[j]), abs(grad_W2[j] - book_W2[j]))
    print(f"{name:<16}{grad_W2[j]:<16.6f}{book_W2[j]:<12.4f}"
          f"{num_W2[j]:<16.6f}{d:.2e}")

print("-" * 70)
assert np.allclose(grad_W1, num_W1, atol=1e-5)
assert np.allclose(grad_W2, num_W2, atol=1e-5)
print("[OK] 세 방법이 모두 일치 — 구현이 확실히 맞다")

### 왜 중앙 차분을 쓰는가

앞에서 $\dfrac{L(w+h) - L(w-h)}{2h}$를 썼다. 이론편 5.1절의 정의는 $\dfrac{L(w+h)-L(w)}{h}$였는데
왜 다를까.

**중앙 차분이 훨씬 정확하기 때문**이다. 두 방식의 오차를 비교해 보자.

In [ ]:
import numpy as np

print("=" * 60)
print("전방 차분 vs 중앙 차분")
print("=" * 60)

target = grad_W1[0, 0]     # 정확한 값 (수식 유도)
print(f"정확한 값: {target:.10f}")
print()
print(f"{'h':<12}{'전방 차분':<22}{'중앙 차분':<22}")
print("-" * 60)

for h in [1e-2, 1e-4, 1e-6, 1e-8]:
    Wp = W1.copy(); Wp[0,0] += h
    Wm = W1.copy(); Wm[0,0] -= h
    L0 = forward_loss(W1, W2, x, y)
    fwd = (forward_loss(Wp, W2, x, y) - L0) / h
    cen = (forward_loss(Wp, W2, x, y) - forward_loss(Wm, W2, x, y)) / (2*h)
    print(f"{h:<12.0e}{abs(fwd-target):<22.2e}{abs(cen-target):<22.2e}")

print("-" * 60)
print("중앙 차분의 오차가 훨씬 작다.")
print()
print("다만 h를 너무 작게 하면 오히려 나빠진다 —")
print("컴퓨터가 아주 작은 수의 뺄셈에서 정밀도를 잃기 때문이다.")
print("보통 1e-5 ~ 1e-7 정도를 쓴다.")

---

## 5. 그래디언트 소실 — 이론편 11.2절

이론편 10.4절 마지막에 층별 그래디언트 크기를 비교한 표가 있었다.
한 층만 거슬러 왔는데도 **37% 수준으로 줄어들었다**는 것이었다.

In [ ]:
import numpy as np

print("=" * 60)
print("층별 그래디언트 크기 (이론편 10.4절 표)")
print("=" * 60)

out_max = np.abs(grad_W2).max()
in_max = np.abs(grad_W1).max()
ratio = in_max / out_max * 100

print(f"{'층':<20}{'그래디언트 최대 절댓값':<26}{'원본 대비'}")
print("-" * 60)
print(f"{'출력층 W2':<20}{out_max:<26.4f}{'기준'}")
print(f"{'입력층 W1':<20}{in_max:<26.4f}{ratio:.0f}%")
print("-" * 60)
print(f"이론편 표: 출력층 0.1614 / 입력층 0.0597 → 약 37%")
assert abs(ratio - 37) < 2, "이론편 값과 다릅니다"
print("[OK] 이론편 10.4절과 일치")
print()
print(f"원인: 시그모이드 도함수 {sig_deriv.mean():.3f} 가 한 번 곱해졌기 때문")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("층이 더 깊어지면 — 이론편 11.2절")
print("=" * 60)

# 시그모이드 도함수의 평균값이 매 층 곱해진다고 가정
avg_deriv = 0.235
depths = np.arange(1, 21)
signal = avg_deriv ** depths

print(f"{'층 수':<10}{'남는 신호':<20}{'비고'}")
print("-" * 60)
for d in [1, 2, 5, 10, 20]:
    s = avg_deriv ** d
    note = ""
    if s < 1e-6: note = "사실상 학습 불가"
    elif s < 1e-3: note = "학습 매우 느림"
    print(f"{d:<10}{s:<20.2e}{note}")
print("-" * 60)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(depths, signal, marker="o", linewidth=2, color="#EA580C",
        label=f"시그모이드 (도함수 평균 {avg_deriv})")
ax.plot(depths, np.ones_like(depths, dtype=float), linewidth=2,
        color="#0D9488", linestyle="--", label="ReLU (도함수 1)")
ax.set_yscale("log")
ax.set_xlabel("층 수")
ax.set_ylabel("앞쪽 층에 도달하는 신호 (상대값)")
ax.set_title("깊이에 따른 그래디언트 감쇠")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print()
print("이것이 이론편 3.3절에서 다룬 '깊은 신경망을 학습시킬 수 없었던' 이유다.")
print("이론편 11.2~11.4절의 ReLU·초기화·정규화가 이 문제를 완화한다.")

---

## 6. 일반화 — 층 개수에 무관한 구현

지금까지는 2층 고정이었다. **층이 몇 개든 동작하는** 형태로 다시 만든다.

핵심은 2절에서 발견한 패턴이다.

| 단계 | 하는 일 |
|---|---|
| 순전파 | 각 층의 입력과 출력을 **저장해 둔다** |
| 역전파 | 뒤에서부터 `delta`를 받아 그래디언트를 만들고, 앞 층으로 넘긴다 |

순전파에서 중간값을 저장하는 이유는, **역전파에서 그것들이 필요하기 때문**이다.
이 때문에 학습 시 메모리가 추론보다 많이 든다(이론편 22.5절).

In [ ]:
import numpy as np


class SimpleNetwork:
    """층 개수에 무관한 신경망 (이론편 10.4절 일반화)"""

    def __init__(self, sizes, seed=0):
        """sizes 예: [2, 4, 3, 1] → 입력2 - 은닉4 - 은닉3 - 출력1"""
        rng = np.random.RandomState(seed)
        self.sizes = sizes
        self.weights = [rng.randn(sizes[i], sizes[i+1]) * 0.5
                        for i in range(len(sizes) - 1)]
        self.biases = [np.zeros((1, sizes[i+1])) for i in range(len(sizes) - 1)]

    def forward(self, X):
        """순전파. 역전파에 필요한 중간값을 함께 저장한다."""
        self.activations = [X]          # 각 층의 출력
        self.z_values = []              # 활성화 이전 값

        a = X
        for W, b in zip(self.weights, self.biases):
            z = a @ W + b
            a = sigmoid(z)
            self.z_values.append(z)
            self.activations.append(a)
        return a

    def backward(self, X, y):
        """역전파. 뒤에서 앞으로 delta를 전달한다."""
        n = len(X)
        grads_W = [None] * len(self.weights)
        grads_b = [None] * len(self.biases)

        # 출력층에서 시작
        a_out = self.activations[-1]
        delta = 2 * (a_out - y) * sigmoid_derivative(a_out) / n

        # 뒤에서 앞으로
        for layer in reversed(range(len(self.weights))):
            a_prev = self.activations[layer]
            grads_W[layer] = a_prev.T @ delta
            grads_b[layer] = delta.sum(axis=0, keepdims=True)

            if layer > 0:
                # 앞 층으로 신호 전달
                delta = (delta @ self.weights[layer].T) * \
                        sigmoid_derivative(self.activations[layer])

        return grads_W, grads_b

    def train(self, X, y, lr=0.5, epochs=3000, record_every=100):
        losses = []
        for ep in range(epochs):
            out = self.forward(X)
            loss = np.mean((out - y) ** 2)
            if ep % record_every == 0:
                losses.append((ep, loss))

            gW, gb = self.backward(X, y)
            for i in range(len(self.weights)):
                self.weights[i] -= lr * gW[i]
                self.biases[i] -= lr * gb[i]
        return losses


# XOR로 검증 (10장과 같은 문제)
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([[0], [1], [1], [0]], dtype=float)

print("=" * 60)
print("일반화된 구현으로 XOR 학습")
print("=" * 60)

for sizes in [[2, 4, 1], [2, 4, 4, 1]]:
    net = SimpleNetwork(sizes, seed=1)
    losses = net.train(X_xor, y_xor, lr=1.0, epochs=8000)
    pred = net.forward(X_xor).ravel()
    final = losses[-1][1]
    ok = "성공" if final < 0.01 else "실패"
    print(f"구조 {str(sizes):<16} 최종손실 {final:.6f}  예측 {pred.round(3)}  {ok}")

print()
print("층을 몇 개 쌓든 같은 코드로 동작한다.")

### 일반화 구현도 검증한다

일반화하면서 실수했을 수 있다. **수치 미분으로 다시 확인한다.**
이것이 05장에서 강조한 습관이다.

In [ ]:
import numpy as np

def check_network_gradients(net, X, y, h=1e-6):
    """일반화 구현의 그래디언트를 수치 미분과 대조한다"""
    net.forward(X)
    grads_W, _ = net.backward(X, y)

    def loss_now():
        return np.mean((net.forward(X) - y) ** 2)

    max_diff = 0.0
    print(f"{'층':<8}{'위치':<12}{'해석적':<16}{'수치':<16}{'차이'}")
    print("-" * 62)

    for layer in range(len(net.weights)):
        W = net.weights[layer]
        # 각 층에서 두 곳만 표본 검사
        for (i, j) in [(0, 0), (min(1, W.shape[0]-1), min(1, W.shape[1]-1))]:
            orig = W[i, j]

            W[i, j] = orig + h;  lp = loss_now()
            W[i, j] = orig - h;  lm = loss_now()
            W[i, j] = orig

            num = (lp - lm) / (2 * h)
            ana = grads_W[layer][i, j]
            diff = abs(num - ana)
            max_diff = max(max_diff, diff)
            print(f"{layer:<8}{f'[{i}][{j}]':<12}{ana:<16.8f}{num:<16.8f}{diff:.2e}")

    print("-" * 62)
    return max_diff


print("=" * 62)
print("일반화 구현 검증 (3층 신경망)")
print("=" * 62)
net = SimpleNetwork([2, 4, 3, 1], seed=0)
md_ = check_network_gradients(net, X_xor, y_xor)

print(f"최대 차이: {md_:.2e}")
assert md_ < 1e-5, "그래디언트 구현에 문제가 있습니다"
print("[OK] 층이 3개인 경우에도 역전파가 정확하다")

---

## 7. 정리

### 확인한 이론편 값 (17개 항목)

| 구분 | 항목 수 | 결과 |
|---|---|---|
| 순전파 | 4개 (z1, a1, z2, L) | 전부 일치 ✓ |
| 역전파 | 13개 (delta, 그래디언트 6개 등) | 전부 일치 ✓ |
| 그래디언트 소실 | 층간 비율 37% | 일치 ✓ |

**3중 검증**으로 확인했다 — 수식 유도 / 이론편 손계산 / 수치 미분이 모두 같았다.

### 기억할 것

| 항목 | 요점 |
|---|---|
| 역전파의 핵심 | 각 층 그래디언트 = **온 신호 × 그 층의 입력** |
| delta 전달 | 뒤에서 앞으로, 활성화 도함수를 곱하며 |
| 중간값 저장 | 순전파에서 저장해야 역전파 가능 → 메모리 사용 |
| 수치 미분 | 중앙 차분, h는 1e-5 ~ 1e-7 |
| 그래디언트 소실 | 시그모이드 도함수(≤0.25)가 층마다 곱해짐 |
| 검증 습관 | 구현을 바꿀 때마다 다시 확인 |

### 왜 직접 만들어 봤는가

다음 장부터는 PyTorch가 이 모든 것을 자동으로 해 준다.
`loss.backward()` 한 줄이면 끝이다.

그럼에도 직접 만든 이유는, **그 한 줄 안에서 무슨 일이 일어나는지 알기 위해서**다.

- 왜 학습에 메모리가 많이 드는가 → 중간값을 저장하기 때문
- 왜 깊은 망이 학습이 안 되는가 → 도함수가 곱해지며 줄어들기 때문
- 왜 ReLU가 나왔는가 → 도함수가 1이라 줄지 않기 때문

이 이해가 있으면 나중에 학습이 안 될 때 어디를 봐야 할지 알게 된다.

### 다음 장

**12. PyTorch 입문 — 텐서와 자동미분** — 방금 만든 것을 PyTorch로 다시 구현한다.
`autograd`가 계산한 그래디언트가 **우리가 손으로 구한 값과 같은지** 확인한다.

### 3중 검증 결과를 그림으로

숫자 표만 보면 "다 맞았다"는 것이 잘 와닿지 않는다.
**세 방법이 얼마나 일치하는지**, 그리고 **층을 거칠 때 그래디언트가 어떻게 줄어드는지**를 그래프로 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 3절에서 구한 값들을 하나로 모은다
labels, analytic, numeric, book = [], [], [], []
for i in range(2):
    for j in range(2):
        labels.append(f"dW1[{i},{j}]")
        analytic.append(grad_W1[i, j])
        numeric.append(num_W1[i, j])
        book.append(book_W1[i, j])
for j in range(2):
    labels.append(f"dW2[{j}]")
    analytic.append(grad_W2[j])
    numeric.append(num_W2[j])
    book.append(book_W2[j])

analytic = np.array(analytic)
numeric = np.array(numeric)
book = np.array(book)

# --- 왼쪽: 세 방법의 값 비교 ---
ax = axes[0]
x = np.arange(len(labels))
w = 0.27
ax.bar(x - w, analytic, w, label="수식 유도", color="#1E40AF")
ax.bar(x, book, w, label="이론편 손계산", color="#0D9488")
ax.bar(x + w, numeric, w, label="수치 미분", color="#EA580C")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=7, rotation=45, ha="right")
ax.set_ylabel("그래디언트 값")
ax.set_title("세 방법이 같은 값을 내는가")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# --- 가운데: 일치도 산점도 ---
ax = axes[1]
ax.scatter(analytic, numeric, s=90, color="#1E40AF",
           edgecolors="white", linewidth=1.2, zorder=3, label="수치 미분")
ax.scatter(analytic, book, s=50, color="#0D9488",
           marker="s", edgecolors="white", linewidth=1, zorder=4, label="이론편 손계산")
lo = min(analytic.min(), numeric.min()) - 0.02
hi = max(analytic.max(), numeric.max()) + 0.02
ax.plot([lo, hi], [lo, hi], "--", color="#DC2626",
        linewidth=1.5, label="완전 일치선")
ax.set_xlabel("수식으로 유도한 값")
ax.set_ylabel("다른 방법의 값")
ax.set_title(f"{len(labels)}개 항목 모두 일치")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- 오른쪽: 층별 그래디언트 크기 ---
ax = axes[2]
mag_W2 = np.abs(grad_W2).mean()
mag_W1 = np.abs(grad_W1).mean()
bars = ax.bar(["출력층 W2", "은닉층 W1"], [mag_W2, mag_W1],
              color=["#1E40AF", "#EA580C"])
for b, v in zip(bars, [mag_W2, mag_W1]):
    ax.text(b.get_x() + b.get_width()/2, v * 1.03, f"{v:.4f}",
            ha="center", fontsize=10)
ratio = mag_W1 / mag_W2 * 100
ax.set_ylabel("그래디언트 평균 크기")
ax.set_title(f"층을 거치면 {ratio:.0f}% 로 줄어든다")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

err_num = np.abs(analytic - numeric).max()
err_book = np.abs(analytic - book).max()

print("왼쪽: 세 색깔의 막대 높이가 같다 = 같은 값")
print()
print(f"가운데: 모든 점이 대각선 위에 놓인다")
print(f"  수치 미분과의 최대 차이 : {err_num:.2e}")
print(f"  이론편 손계산과의 최대 차이 : {err_book:.2e}")
print("  수치 미분은 근사이므로 완전히 0이 되지는 않는다.")
print()
print(f"오른쪽: 한 층 거칠 때마다 그래디언트가 {ratio:.0f}% 수준으로 준다")
print("  층이 10개면 이 비율이 10번 곱해진다 — 이론편 11.2절의 소실 문제")
print(f"  {ratio/100:.2f}^10 = {(ratio/100)**10:.6f}")